#### M8B1_AI1: AWS MLOPs
#### Autor: Leandro Gutierrez
#### Este documento intenta dar respuesta a la actividad 1 propuesta en el Modulo **Gestión de proyectos**. En él se describirán cada uno de los enunciados postulados y el desarrollo realizado.

# Enunciado
Sabiendo que tienes acceso a La solución [AWS MLOps[1]](https://github.com/awslabs/aws-mlops-framework) que es un Framework que te ayuda a optimizar y aplicar las mejores prácticas de arquitectura para la producción de modelos de aprendizaje automático (ML), se te pide investigar como data scientist y proponer los pasos a seguir para poner en producción el [case base reasoning[2]](https://arxiv.org/abs/2104.00409). 

Basarse en la referencia [1] y [2] y hacer una investigación de los pasos a seguir para la puesta en producción el case base reasoning.

Te pedimos resultado, conclusión y justificación a través de un documento en formato Colab que responda a las preguntas planteadas en esta tarea teórica.

# Solución
En este trabajo práctico abordaremos los aspectos clave para el desarrollo y puesta en producción de un servicio clasificador basado en modelos de Machine Learning. Desplegar en producción un servicio de estas características no es tarea trivial y el abordaje dependerá del contexto donde se implemente. Es así que solución debe contemplar el capital humano con el que se cuente, el conjunto de procesos existentes y de la infraestrucutra disponible.

A modo académico se asumirá el uso de Amazon Web Service como servicio de infraestructura en la nube. Sin embargo la solución propuesta al ser concebida como conjunto de bloques funcionales podría ser interoperable entre diferentes plataformas. Se brindarán los conceptos básicos y deseables de cada etapa del desarrollo y despligue, mientras se comparan soluciones on-premise, híbridas y otras totalmente orientadas a servicios cloud.

## Etapas del desarrollo
Para dimensionar y reducir la complejidad total del problema lo dividiremos en 4 etapas que pueden comprenderse como las básicas dentro del desarrollo de cualquier solución de Machine Learning:

- Preparación y exploración inicial
- Desarrollo
- Despligue
- Mejora continua

## Preparación de los datos y exploración inicial
Sin datos no existe Machile Learning, por lo que es requisito indispensable para comenzar con nuestras taréas como científicos de datos contar con la información mínima para encausar nuestra investigación y posterior desarrollo. En esta primera etapa detallaremos los setups o configuraciones típicas al momento de realizar una primera aproximación exploratória de la solución final.

### Entornos de trabajo
Es fundamental en todo proyecto contar con un entorno adecuado para experimentación. La dinámica de trabajo dependerá de la infraestructura, tanto del hardware disponible para los desarrolladores y analistas, como de la infraestructura en servicios de cloud. A continuación mostraremos tres opciones de trabajo:

#### Setup Local
Esta primera opción requiere  mayor conocimiento por parte del desarrollador ya que necesitará manejar por cuenta própia la instalación de las dependencias para trabajar:
- IDE: interfaz de desarrollo, permite manipular los archivos de trabajo. Los ejemplos mas conocidos VS-Code, R-studio, entre otros.
- Git: repositorio de codigo, permite trackear cada evolución en el desarrollo y trabajar de manera colaborativa. Ejemplos mas conocidos: GitHub, CodeCommit o Bitbucket
- Jupyter Server: servicio que nos permite gestionar nuestras notebooks, implementa la comunicación con diferentes kernels y probee una interfaz web para la creación y ejecución de las notebooks.
- Los datos se asumen accesibles: el conjunto de datos se puede suponer disponible en una fuente externa.

![Opcion 1](/Users/lgutierrez/Proyectos/master/M8/images/local.png)

#### Setup Hibrido
Esta opción requiere un nivel de conocimientos en cuanto a infraestructura un poco más profundo, ya el equipo de MLOps se responsabiliza del setup inicial de los servicios necesarios para que los desarrolladores y científicos puedan realizar su trabajo. Esta opción permite liberar de responsabilidades al desarrollador utilizando servicios gestionados por la empresa, los mismos puede derivar de infraestructura propia de la compañia o servicios de cloud contratados, pero totalmente gestionado por el equipo de MLOps.

![Opcion 2](/Users/lgutierrez/Proyectos/master/M8/images/remoto.png)

#### Setup 100% Cloud
Esta última opción completamente gestionada por Amazon Web Service hace uso del servicio [SageMaker Studio](https://us-east-1.console.aws.amazon.com/sagemaker/home?region=us-east-1#/studio-landing), el cual probee un entorno de exploración y desarrollo listo para trabajar. Permite trabajar con notebooks Jupyter y RStudio de manera completamente remota y sin preocupar al equipo por el despligue de los componentes necesarios. SageMaker implementa las conexiones con las principales fuentes de información disponibles dentro de la plataforma AWS y fuera de ella.

![Opcion 3](/Users/lgutierrez/Proyectos/master/M8/images/sage.png)

### Recolección y preprocesado de datos
Como dijimos anteriormente sin datos no hay modelos, por lo que es necesario contar con una correcta gestión de la información dentro de nuestro proyecto. Al igual que para el setup de la etapa exploratoria, existen también múltiples posibilidades de configuración para esta infraestructura, pero en este caso es común relegar al equipo de MLOps la puesta a punto, ya que es común contar con grandes volúmenes de información lo que hace dificil el manejo de forma local de la misma. En este caso plantearemos dos posibilidades, la primera totalmente administrada por el equipo de trabajo y una segunda configuración totalmente gestionada por AWS.

#### Opción auto-administrado
En los diagramas anteriores tratamos de abstraer la complejidad vinculada a la recolección y disponibilización de los datos. Este bloque funcional engloba todos los servicios que interactúan durante el movimiento de información entre diferentes puntos dentro de nuestra infraestructura. Abarca las diferentes base de datos (OLTP, OLAP, estructuradas, no estructuradas, etc), asi como tambien los pipelines de sincronización en tiempo real y los trabajos programados que se utilizan para recolectar y preprocesar información.

![Data](/Users/lgutierrez/Proyectos/master/M8/images/data.png)

Los datos son la pieza fundamental para el correcto trabajo de los científicos, sin ella el trabajo ni siquiera podría comenzar. Es por ello fundamental tener claro desde donde se extraerá la información, con que perioricidad y como se la dispondrá para su consulta.

Existen diferentes maneras mover información de un punto a otro. Entre los mecanismos de sincronización destacan los orientados a tiempo real y los enfocados en tareas batch. La elección dependerá del tipo de problema que enfrentemos, el atraso máximo permitido por el requerimiento y de los recursos con los que se cuenten:

- **Spark** se posiciona como el framework de manipulación de datos por excelencia, se integra muy bien con el ecosistema **Hadoop**, haciendo uso de **HDFS** y **YARN**.
- **Airflow** se ecuentra entre los orquestadores de tareas más conocidos y utilizados, permite programar y monitorear tareas de todo tipo, tanto de sincronización de datos, como de checkeos de keep-alive, entre otros.
- **Kafka** uno de los mecanismos de sincronización en tiempo real mas demandado en la actualidad, presenta una versión distribuida siendo facilmente escalable y resiliente ante alta carga.

En la ilustación anterior hacemos referencia al concepto de **datalake**, como aquel componente que nos permite almacenar indistintamente infomación estructurada como no estructurada, proveniente de multiples fuentes y cuyo fin es el de proveer información para análisis. También se comprende capaz de albergar y gestionar los modelos que se utilicen durante el desarrollo.

![Lake](/Users/lgutierrez/Proyectos/master/M8/images/lake.png)

#### Opción Cloud
AWS ofrece estas mismas funcionalidades totalmente gestionadas por la plataforma, es así que podriamos nombrar a:

- Amazon EMR: servicio que permite manipular y gestionar cargas de trabajos para grandes volumenes de información. Permite al equipo abstraerse de la complejidad de dimensionar y escalar dinámincamente los componentes. Permite crear cargas de trabajo utilizando **Spark**, **Hive** y **Presto**.
- Amazon Glow: sericio serverless de AWS que permite crear catálogos de datos de manera sencilla, además permite crear, monitorear y ejectar tareas de canalización de datos. La solucíon implementa conectividad con mas de 100 fuentes de datos diferentes.
- Amazon MWAA: permite implementar **Airflow** a gran escala sin lidiar con la infraestructura subycente.
- Amazon S3: Amazon Simple Storage Service es el servicio por excelencia para crear data lakes.
- AWS Step Functions: permite crear canalizaciones ETL y ELT de manera sencilla dentro de la plataforma AWS.

![s3](/Users/lgutierrez/Proyectos/master/M8/images/s3.png)


## Desarrollo
Una vez que ya tenemos los entornos de exploración listos y los pipelines de datos ajustados a nuestro caso de uso, nos queda determinar como el equipo de desarrollo trabajará de manera colaborativa para llegar a satisfacer los requisitos planteados. Nuevamente existen múltiples maneras de llevar a cabo esta etapa, ya sea utilizando herramientas open-source o soluciones 100% cloud. Lo que buscaremos es una via que nos permita definir, entrenar y evaluar nuestros modelos de manera sencilla y clara.

#### Opción de código abierto
[MLFlow](https://mlflow.org/docs/latest/index.html) es la herramienta de codigo abierto quizas mas utilizada de momento, la misma nos permite tener un seguimiento completo de nuestros proyectos de Machine Learning a lo largo de su ciclo de vida. El mismo se basa en trackear hiperparámetros de nuestros modelos, métricas de tiempo y precisión, así como otros metadatos de utilidad. Además brinda una convención de trabajo y organización para los proyectos de ML. Cuenta con un registro centralizado de modelos, permitiendo el trabajo colaborativo y versionado de los modelos desarrollados. Cuenta con una interfaz de usuario y con diferentes APIs para acceder a las distintas funcionalidades.

![MLFlow](/Users/lgutierrez/Proyectos/master/M8/images/mlflow.png)

#### Opción Cloud
Como mencionamos anteriormente AWS cuenta con [Amazon SageMaker AI](https://aws.amazon.com/es/sagemaker-ai/) como punto fundamental en el desarrollo de modelos de Machine Learning, el mismo cuenta con SageMaker Studio, una interfaz de desarrollo basada en Jupyter que permite desarrollar y ejecutar notebooks de manera sencilla. Maneja de manera automatica el escalado de los recursos necesarios para el trabajo del equipo de desarrollo. Permite la gestión eficiente de todo el ciclo del desarrollo, desde en entrenamiento hasta la puesta en producción de la solción.Implementa conectividad con múltiples fuentes de datos y ofrece acceso a modelos pre-entrenados.

Junto con SageMaker se integran CodeCommit como repositorio de código, Model Registry (EMR) como repositorio de modelos y Amazon Elastic Container Registry (ECR) el cual permite gestionar contenedores de nuestros servicios.

![Sage](/Users/lgutierrez/Proyectos/master/M8/images/sage.png)


## Despliegue
El despliegue es una de las etapas críticas en el ciclo de vida de cualquier solución de Machine Learning (ML). En esta fase, el modelo entrenado se transforma en un servicio accesible que puede ser utilizado por sistemas o usuarios finales. A continuación, exploramos dos enfoques principales: soluciones de código abierto y servicios cloud totalmente gestionados.

### Opción de código abierto
[Kubernetes](https://kubernetes.io/es/docs/concepts/overview/) es una plataforma de orquestación de contenedores que permite desplegar y escalar aplicaciones, incluidos servicios de ML. El modelo puede empaquetarse como un microservicio en un contenedor Docker y desplegarse en un clúster de Kubernetes. 

[MLFlow Models](https://mlflow.org/docs/latest/models.html) permite empaquetar el modelo ya entrenado y desplegarlo como un servicio REST mediante comandos simples. Esto permite servir predicciones a sistemas externos y facilita la integración con otras aplicaciones.

[Jenkins](https://www.jenkins.io/) u otras herramientas de integración y despliegue continuo (CI/CD) pueden ser utilizada para automatizar el proceso de despliegue. También se podían utilizar [Github Actions](https://github.com/features/actions) para buildear, testear y desplegar nuestra solución.

### Opción Cloud

Nuevamente AWS ofrece un conjunto de servicios altamente integrados que simplifican el despliegue de modelos. Combinando SageMaker, ECR, AWS Lambda, Amazon Event bridge y Amazon API Gateway podríamos ofrecer un pipeline de despligue completo para nuestra implementación.

![aws](/Users/lgutierrez/Proyectos/master/M8/images/aws-simple.png)

## Mejora continua
Una vez desplegado nuestro servicio necesitaremos monitorear, mantener y mejroar el rendimiento del mismo, para ello es clave seguir buenas prácticas para esta etapa, para ello se pueden utilizar diferentes herramientas disponibles en el mercado.

#### Monitoreo
- MLFlow Tracking: Permite registrar métricas y eventos del modelo en producción, proporcionando una trazabilidad clara de los cambios y su impacto.
- Amazon CloudWatch: Monitoriza métricas en tiempo real como latencia, errores de predicción y uso de recursos del endpoint.
- Amazon SageMaker Model Monitor: Detecta drift en los datos de entrada y salida del modelo, generando alertas automáticas cuando se exceden ciertos umbrales.

## Conclusión

Poner en producción una solución basada en Case Base Reasoning requiere una estrategia bien definida que aborde cada etapa del ciclo de vida del modelo. La elección entre soluciones de código abierto o servicios gestionados depende de las necesidades específicas del proyecto y los recursos disponibles. Sin embargo, AWS proporciona una solución integral que simplifica el desarrollo, despliegue y mejora continua, reduciendo el tiempo de implementación y asegurando escalabilidad. Esto justifica ampliamente su uso en contextos empresariales donde la rapidez y confiabilidad son prioritarias.